   Yes, roughly, but the precise term here is FLOW MATCHING, not standard
   diffusion denoising.

   What the code is doing:

   - sample Gaussian noise `x_0 ~ N(0, I)`
   - take the real target action `x_1`
   - interpolate between them:
      - `x_t = (1 - t) * x_0 + t * x_1`
   - train the model to predict the velociy field:
      - `v = x_1 = x_0`

   That is explicit in `.../aic_training/training/losses.py`.

   So the model is not directly predicting actions in one shot during training.
   It learns a conditional vector field that tells you how to move from noise
   toward the action distribution, conditioned on:
      - encoded image
      - encoded robot/task state
      - timestep embedding `t`

   Architecturally, the denoiser-ish core is a DiT style transformer in
   `./aic_training/models/flow_matching.py`:
      - image encoder
      - state encoder
      - timestep embedding
      - transformer blocks
      - output projection to action-velocity field

   Then at inference time it starts from random noise and integrates forward 
   with Euler ODE steps using `sample()`, instead of doing a classic diffusion
   reverse-noising schedule.

   So the clean distinction is:
      - diffusion: usually learns score/noise prediction over a noising schedule   


-- Diffusion Transformers (DiT) represent a class of generative models that
   replace the traditional convolutional U-Net backbone in diffusion models with
   a TRANSFORMER ARCHITECTURE. Developed by... DiT models operate on latent 
   images (patchified tokens) rather than raw pixels, enabling better 
   scalability and modeling of long-range dependencies in image generation.

CORE ARCHITECURE OF DiT
   The DiT architecture is based heavily on the VISION TRANSFORMER (ViT), with
   key adjustments for diffusion tasks:
   - PATCHIFY: The input latent image is divided into fixed-size patches and
     projected into tokens (similar to how BERT processes text).
   - TRANSFORMER BLOCKS: A sequence of standard transformer blocks processes
     these tokens.
   - CONDITIONING (adaLN-Zero): To incorporate conditional inputs like timesteps
     ($t$) or class labels ($c$), DiT uses adaptive layer normalisation 
     (adaLN-Zero), which modulates token activations. This approach is more 
     efficient than standard cross-attention for certain tasks.
   - UNPATCHIFY: The final sequence of tokens is constructured back into the 
     latent space format.

KEY ADVANTAGES
   - SCALABILITY: DiT models show that as Gflops (forward pass complexity)
     increase -- through higher depth, width, or token count--the model's 
     performance (FID) improves predictably.
   - GLOBAL CONTEXT: Unlike U-Nets, which rely on convolutions (local receptive
     fields), DiTs use self-attention, allowing them to capture global d
     dependencies across the entire image more effectively.
   - PERFORMANCE: Larger DiT models (e.g., DiT-XL/2) outperform previous 
     diffusion models on class-conditional ImageNet benchmarks.

COMMON DiT VARIANTS AND USE CASES
   - IMAGE GENERATION: The original DIT models designed for class-conditional
     image synthesis (e.g., 256x256, 512x512).
   - TEXT-TO-IMAGE (T2I): Used in high-performance generative models like 
     PixArt-alpha and STABLE DIFFUSION 3, ...
   - VIDEO GENERATION: SoRA (OpenAI) utilises DiT architectures to handle the 
     high-dimensional data of video over time.
   - DiT-3D: Adapts the DiT architecture for 3D point cloud generation, 
     demonstrating superior performance on ShapeNet.

   DiT is rapidly becoming the new standard for diffusion model backbones, 
   shifting the focus from CNNs to transformers in the gen-AI space.     



---
-- In transformer-based robotics and computer vision, LATENT IMAGES (or latent
   representations) are compact, compressed numerical representations of raw
   pixel data. Instead of processing high-resolution images pixel-by-pixel--which
   is computationally expensive--transformers work with these "smaller", 
   semantically rich representations, usually generated by an encoder (like a 
   Vector Quantised Variational Autoencoder, VQ-VAE, or GAN).

   Key aspect of latent images in AI and robotics include:
      - DIMENSIONALITY REDUCTION: They act as a bottleneck, reducing image
        dimensions (e.g., 16x smaller) while retaining essential structural and
        semantic information.
      - VECTOR QUANTISATION (VQ-GAN): In many transformer model (like Latent 
        Transformer Models or Latte), the encoder breaks images into patches,
        transforms them into latent vectorrs, and maps them to a finite
        "codebook" of feature vectors. 
      - TRANSFORMER INPUT: The transformer processes these compressed latents as
        a 1D sequnece (treating them like "tokens" in a language model) to learn
        spatial or temporal dependencies.
      - GENERATION/DECODING: After the transformer predicts future or generated
        latents, a decoder translates this low-dimensional, compressed 
        representation back into a full-sized, pixel-space image. 

Use in Robotics & AI:
   In robotic vision, latent representations are used to predict future scenes
   (video prediction) or to understand the environment while maintaining an 
   internal estimate of the robot's state. This approach allows for faster and
   more memory-efficient processing compared to operating directly on pixel
   space.     



- Might want to look at how the math works for latent images
   - especially with  VQ-GAN for example. (I want to understand how do they turn
     arbitrary images (which could vary in size, lighting, and resolution) (and
     how it's trained to do this too in the optimiser process), into feature
     vectors... and how the "codebook" even knos what and how to map to what...
     is this like how tokenisers are made in transformers? where tokenisers are
     calculated before pre-training even starts?)

---

   Diffusion Transfromer DiT style architectures are transforming robotics by
   replacing traditional U-Net backbones in diffusion models with 